In [5]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

# Read the original healthcare Encounter table
df = spark.table("dbo.`appointments encounters`")

# Count all rows
total_rows = df.count()

# Count distinct Encounter IDs
unique_appointments = (
    df.select("Encounter_ID")
      .distinct()
      .count()
)

# Count completely identical rows
distinct_rows = df.distinct().count()

print("Total rows:", total_rows)
print("Unique appointments:", unique_appointments)
print("Distinct complete rows:", distinct_rows)
print("Exact duplicate rows:", total_rows - distinct_rows)


# Check whether the Lakehouse is connected
spark.sql("SHOW TABLES IN dbo").show()


# Read the original Encounter table
df = spark.table("dbo.`appointments encounters`")

# Count records
total_rows = df.count()

# Count unique appointments
unique_appointments = (
    df.select("Encounter_ID")
      .distinct()
      .count()
)

# Count distinct complete rows
distinct_rows = df.distinct().count()

# Calculate exact duplicate rows
exact_duplicates = total_rows - distinct_rows

print("Total rows:", total_rows)
print("Unique appointments:", unique_appointments)
print("Distinct complete rows:", distinct_rows)
print("Exact duplicate rows:", exact_duplicates)


# Read the original Encounter table
df = spark.table("dbo.`appointments encounters`")

# Remove completely identical rows
df_clean = df.dropDuplicates()

# Verify the cleaned dataset
print("Original rows:", df.count())
print("Clean rows:", df_clean.count())
print(
    "Unique appointments:",
    df_clean.select("Encounter_ID").distinct().count()
)

# Save as a new Lakehouse Delta table
df_clean.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable("dbo.FactAppointments")

print("FactAppointments created successfully!")



# Verify the existing cleaned table
df_check = spark.table("dbo.factappointments")

print("Total rows:", df_check.count())

print(
    "Unique appointments:",
    df_check.select("Encounter_ID").distinct().count()
)


# Show all tables in the Lakehouse
spark.sql("SHOW TABLES IN dbo").show(
    truncate=False
)


# Show all tables in the Lakehouse
spark.sql("SHOW TABLES IN dbo").show(
    truncate=False
)


# Create clean provider dimension
df_dimprovider = df_providers.dropDuplicates()

# Save without overwriting an existing table
df_dimprovider.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable("dbo.DimProvider")

print("DimProvider created successfully!")
print("Total providers:", df_dimprovider.count())


df_providers = spark.table("dbo.providers")

print("Total rows:", df_providers.count())
print("Distinct complete rows:",
      df_providers.distinct().count())

print("Exact duplicate rows:",
      df_providers.count() -
      df_providers.distinct().count())




StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 7, Finished, Available, Finished, False)

Total rows: 10018
Unique appointments: 10000
Distinct complete rows: 10000
Exact duplicate rows: 18
+--------------------+--------------------+-----------+
|           namespace|           tableName|isTemporary|
+--------------------+--------------------+-----------+
|Healthcare_NoShow...|       CodeDiagnosis|      false|
|Healthcare_NoShow...|appointments enco...|      false|
|Healthcare_NoShow...|          facilities|      false|
|Healthcare_NoShow...|    factappointments|      false|
|Healthcare_NoShow...|           providers|      false|
+--------------------+--------------------+-----------+

Total rows: 10018
Unique appointments: 10000
Distinct complete rows: 10000
Exact duplicate rows: 18
Original rows: 10018
Clean rows: 10000
Unique appointments: 10000


AnalysisException: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `chimcobldhq2ai35c5m78q33c5p6anqedt9mgrrnbt0msobcf5q6iorj4l46aobcehk66obiclfkoobbclk6utbjckim8ojf`.`FactAppointments` because it already exists.
Choose a different name, drop or replace the existing object, or add the IF NOT EXISTS clause to tolerate pre-existing objects. SQLSTATE: 42P07

In [7]:

from pyspark.sql import functions as F

# Read raw provider records
df_providers = spark.table("dbo.providers")

# Remove exact duplicates
df_dimprovider = df_providers.dropDuplicates()

# Validate keys
missing_ids = df_dimprovider.filter(
    F.col("Provider_ID").isNull() |
    (F.trim(F.col("Provider_ID").cast("string")) == "")
).count()

total = df_dimprovider.count()
unique = df_dimprovider.select(
    "Provider_ID"
).distinct().count()

print("Missing IDs:", missing_ids)
print("Clean rows:", total)
print("Unique Provider IDs:", unique)

if missing_ids != 0 or total != unique:
    raise ValueError("Provider data needs further review.")

# Save cleaned dimension table
df_dimprovider.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable("dbo.DimProvider")

print("DimProvider created successfully!")


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 9, Finished, Available, Finished, False)

Missing IDs: 0
Clean rows: 120
Unique Provider IDs: 120


AnalysisException: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `chimcobldhq2ai35c5m78q33c5p6anqedt9mgrrnbt0msobcf5q6iorj4l46aobcehk66obiclfkoobbclk6utbjckim8ojf`.`DimProvider` because it already exists.
Choose a different name, drop or replace the existing object, or add the IF NOT EXISTS clause to tolerate pre-existing objects. SQLSTATE: 42P07

In [10]:
df_providers = spark.table("dbo.providers")

StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 12, Finished, Cancelled, Cancelled, False)

In [12]:

df_providers = spark.table("dbo.providers")

print("Total rows:", df_providers.count())

print(
    "Distinct complete rows:",
    df_providers.distinct().count()
)

print(
    "Exact duplicate rows:",
    df_providers.count() -
    df_providers.distinct().count()
)


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 14, Finished, Available, Finished, False)

Total rows: 121
Distinct complete rows: 120
Exact duplicate rows: 1


In [14]:

from pyspark.sql import functions as F

df_providers = spark.table("dbo.providers")

# Remove exact duplicate rows in memory
df_clean = df_providers.dropDuplicates()

# Check missing Provider IDs
missing_ids = df_clean.filter(
    F.col("Provider_ID").isNull() |
    (F.trim(F.col("Provider_ID").cast("string")) == "")
).count()

# Check repeated IDs after removing exact duplicates
conflicting_ids = (
    df_clean
    .groupBy("Provider_ID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Clean provider rows:", df_clean.count())
print("Missing Provider IDs:", missing_ids)
print("Conflicting Provider IDs:", conflicting_ids)


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 16, Finished, Available, Finished, False)

Clean provider rows: 120
Missing Provider IDs: 0
Conflicting Provider IDs: 0


In [16]:

# Confirm data quality before saving
assert missing_ids == 0, "Missing Provider IDs found"
assert conflicting_ids == 0, "Conflicting Provider IDs found"

# Create a new dimension table
df_clean.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable("dbo.DimProvider")

print("DimProvider created successfully!")
print("Total providers:", df_clean.count())




StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 18, Finished, Available, Finished, False)

AnalysisException: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `chimcobldhq2ai35c5m78q33c5p6anqedt9mgrrnbt0msobcf5q6iorj4l46aobcehk66obiclfkoobbclk6utbjckim8ojf`.`DimProvider` because it already exists.
Choose a different name, drop or replace the existing object, or add the IF NOT EXISTS clause to tolerate pre-existing objects. SQLSTATE: 42P07

In [18]:

# Inspect the provider dimension
df_provider = spark.table("dbo.dimprovider")

# Show its structure
df_provider.printSchema()

# Preview records
display(df_provider.limit(5))


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 20, Finished, Cancelled, Cancelled, False)

root
 |-- Provider_ID: string (nullable = true)
 |-- Provider_Name: string (nullable = true)
 |-- Specialty: string (nullable = true)
 |-- Primary_Facility_ID: string (nullable = true)
 |-- Credential: string (nullable = true)
 |-- Employment_Status: string (nullable = true)
 |-- FTE: string (nullable = true)
 |-- Start_Date: string (nullable = true)



Py4JJavaError: An error occurred while calling z:com.microsoft.spark.notebook.visualization.display.getDisplayResultForIPython.
: org.apache.spark.SparkException: [SPARK_JOB_CANCELLED] Job 191 cancelled part of cancelled job group 20 SQLSTATE: XXKDA
	at org.apache.spark.errors.SparkCoreErrors$.sparkJobCancelled(SparkCoreErrors.scala:222)
	at org.apache.spark.scheduler.DAGScheduler.handleJobCancellation(DAGScheduler.scala:3428)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleJobGroupCancelled$7(DAGScheduler.scala:1331)
	at scala.runtime.java8.JFunction1$mcVI$sp.apply(JFunction1$mcVI$sp.scala:18)
	at scala.collection.mutable.HashSet$Node.foreach(HashSet.scala:450)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:376)
	at org.apache.spark.scheduler.DAGScheduler.handleJobGroupCancelled(DAGScheduler.scala:1331)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.doOnReceive(DAGScheduler.scala:3751)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onRe

In [22]:

from pyspark.sql import functions as F

# Read the existing dimension
df_provider = spark.table("dbo.dimprovider")

# Check whether PRV-9999 already exists
existing = df_provider.filter(
    F.col("Provider_ID") == "PRV-9999"
).count()

print("Existing PRV-9999 records:", existing)

if existing == 0:

    # Create a placeholder with the same schema
    unknown_provider = spark.createDataFrame(
        [(
            "PRV-9999",
            "Unknown Provider",
            "Unknown",
            None,
            None,
            "Unknown",
            None,
            None
        )],
        schema=df_provider.schema
    )

    # Append to the existing Delta table
    unknown_provider.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("dbo.dimprovider")

    print("Unknown Provider added successfully!")

else:
    print("PRV-9999 already exists. No changes made.")


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 24, Finished, Available, Finished, False)

Existing PRV-9999 records: 1
PRV-9999 already exists. No changes made.


In [23]:

# Reload the updated table
df_provider_updated = spark.table("dbo.dimprovider")

# Count total providers
print("Total providers:", df_provider_updated.count())

# Count unique provider IDs
print(
    "Unique provider IDs:",
    df_provider_updated.select("Provider_ID")
    .distinct()
    .count()
)

# Display the placeholder record
display(
    df_provider_updated.filter(
        F.col("Provider_ID") == "PRV-9999"
    )
)


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 25, Finished, Available, Finished, False)

Total providers: 121
Unique provider IDs: 121


SynapseWidget(Synapse.DataFrame, f8876c0b-9872-405f-8296-e65d30201269)

In [25]:

# Read the original facilities table
df_facilities = spark.table("dbo.facilities")

# Preview 10 facility records
display(df_facilities.limit(10))

# Display column names and data types
df_facilities.printSchema()

# Count original records
print("Total facility rows:", df_facilities.count())


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 27, Finished, Cancelled, Cancelled, False)

Py4JJavaError: An error occurred while calling z:com.microsoft.spark.notebook.visualization.display.getDisplayResultForIPython.
: org.apache.spark.SparkException: [SPARK_JOB_CANCELLED] Job 221 cancelled part of cancelled job group 27 SQLSTATE: XXKDA
	at org.apache.spark.errors.SparkCoreErrors$.sparkJobCancelled(SparkCoreErrors.scala:222)
	at org.apache.spark.scheduler.DAGScheduler.handleJobCancellation(DAGScheduler.scala:3428)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleJobGroupCancelled$7(DAGScheduler.scala:1331)
	at scala.runtime.java8.JFunction1$mcVI$sp.apply(JFunction1$mcVI$sp.scala:18)
	at scala.collection.mutable.HashSet$Node.foreach(HashSet.scala:450)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:376)
	at org.apache.spark.scheduler.DAGScheduler.handleJobGroupCancelled(DAGScheduler.scala:1331)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.doOnReceive(DAGScheduler.scala:3751)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onRe

In [27]:

from pyspark.sql import functions as F

# Count missing or blank values in each column
missing_values = df_facilities.select([
    F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_facilities.columns
])

display(missing_values)


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8b803bad-d744-4b4e-a4d3-a23701061980)

In [29]:

# Count original records
total_rows = df_facilities.count()

# Count distinct complete records
distinct_rows = df_facilities.distinct().count()

# Count distinct facility IDs
unique_facilities = (
    df_facilities
    .select("Facility_ID")
    .distinct()
    .count()
)

print("Total facility rows:", total_rows)
print("Distinct complete rows:", distinct_rows)
print("Unique Facility IDs:", unique_facilities)
print("Exact duplicate rows:", total_rows - distinct_rows)



StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 31, Finished, Available, Finished, False)

Total facility rows: 26
Distinct complete rows: 25
Unique Facility IDs: 25
Exact duplicate rows: 1


In [31]:

# Find facility IDs appearing more than once
duplicate_facilities = (
    df_facilities
    .groupBy("Facility_ID")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

display(duplicate_facilities)


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 33, Finished, Cancelled, Cancelled, False)

Py4JJavaError: An error occurred while calling z:com.microsoft.spark.notebook.visualization.display.getDisplayResultForIPython.
: org.apache.spark.SparkException: [SPARK_JOB_CANCELLED] Job 245 cancelled part of cancelled job group 33 SQLSTATE: XXKDA
	at org.apache.spark.errors.SparkCoreErrors$.sparkJobCancelled(SparkCoreErrors.scala:222)
	at org.apache.spark.scheduler.DAGScheduler.handleJobCancellation(DAGScheduler.scala:3428)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleJobGroupCancelled$7(DAGScheduler.scala:1331)
	at scala.runtime.java8.JFunction1$mcVI$sp.apply(JFunction1$mcVI$sp.scala:18)
	at scala.collection.mutable.HashSet$Node.foreach(HashSet.scala:450)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:376)
	at org.apache.spark.scheduler.DAGScheduler.handleJobGroupCancelled(DAGScheduler.scala:1331)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.doOnReceive(DAGScheduler.scala:3751)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onRe

In [33]:

# Remove exact duplicate rows in memory
df_facility_clean = df_facilities.dropDuplicates()

# Count missing or blank Facility IDs
missing_ids = df_facility_clean.filter(
    F.col("Facility_ID").isNull() |
    (F.trim(F.col("Facility_ID").cast("string")) == "")
).count()

# Count Facility IDs with conflicting records
conflicts = (
    df_facility_clean
    .filter(
        F.col("Facility_ID").isNotNull() &
        (F.trim(F.col("Facility_ID").cast("string")) != "")
    )
    .groupBy("Facility_ID")
    .count()
    .filter(F.col("count") > 1)
)

print("Clean facility rows:", df_facility_clean.count())
print("Missing Facility IDs:", missing_ids)
print("Conflicting Facility IDs:", conflicts.count())

# Display conflicting records if present
display(
    df_facility_clean.join(
        conflicts.select("Facility_ID"),
        "Facility_ID",
        "inner"
    )
)


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 35, Finished, Cancelled, Cancelled, False)

Py4JJavaError: An error occurred while calling o26643.count.
: org.apache.spark.SparkException: [SPARK_JOB_CANCELLED] Job 260 cancelled part of cancelled job group 35 SQLSTATE: XXKDA
	at org.apache.spark.errors.SparkCoreErrors$.sparkJobCancelled(SparkCoreErrors.scala:222)
	at org.apache.spark.scheduler.DAGScheduler.handleJobCancellation(DAGScheduler.scala:3428)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleJobGroupCancelled$7(DAGScheduler.scala:1331)
	at scala.runtime.java8.JFunction1$mcVI$sp.apply(JFunction1$mcVI$sp.scala:18)
	at scala.collection.mutable.HashSet$Node.foreach(HashSet.scala:450)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:376)
	at org.apache.spark.scheduler.DAGScheduler.handleJobGroupCancelled(DAGScheduler.scala:1331)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.doOnReceive(DAGScheduler.scala:3751)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onReceive(DAGScheduler.scala:3724)
	at org.apache.spark.scheduler.DAGSc

In [35]:

# Safety checks
assert missing_ids == 0, "Missing Facility IDs found"
assert conflicts.count() == 0, "Conflicting Facility IDs found"

# Confirm each Facility ID is unique
total = df_facility_clean.count()
unique = (
    df_facility_clean
    .select("Facility_ID")
    .distinct()
    .count()
)

assert total == unique, "Facility IDs are not unique"

# Create DimFacility as a Delta table
df_facility_clean.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable("dbo.DimFacility")

print("DimFacility created successfully!")
print("Total facilities:", total)


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 37, Finished, Available, Finished, False)

AnalysisException: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `chimcobldhq2ai35c5m78q33c5p6anqedt9mgrrnbt0msobcf5q6iorj4l46aobcehk66obiclfkoobbclk6utbjckim8ojf`.`DimFacility` because it already exists.
Choose a different name, drop or replace the existing object, or add the IF NOT EXISTS clause to tolerate pre-existing objects. SQLSTATE: 42P07

In [37]:

from pyspark.sql import functions as F

# Read the existing fact table
df_fact = spark.table("dbo.factappointments")

# Standardize Facility_ID
df_fact_clean = df_fact.withColumn(
    "Facility_ID",
    F.upper(F.trim(F.col("Facility_ID")))
)

# Confirm row counts remain unchanged
print("Original appointments:", df_fact.count())
print("Cleaned appointments:", df_fact_clean.count())

# Check standardized facility IDs
display(
    df_fact_clean.select("Facility_ID")
    .distinct()
    .orderBy("Facility_ID")
)


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 39, Finished, Cancelled, Cancelled, False)

Py4JJavaError: An error occurred while calling o28314.count.
: org.apache.spark.SparkException: [SPARK_JOB_CANCELLED] Job 295 cancelled part of cancelled job group 39 SQLSTATE: XXKDA
	at org.apache.spark.errors.SparkCoreErrors$.sparkJobCancelled(SparkCoreErrors.scala:222)
	at org.apache.spark.scheduler.DAGScheduler.handleJobCancellation(DAGScheduler.scala:3428)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleJobGroupCancelled$7(DAGScheduler.scala:1331)
	at scala.runtime.java8.JFunction1$mcVI$sp.apply(JFunction1$mcVI$sp.scala:18)
	at scala.collection.mutable.HashSet$Node.foreach(HashSet.scala:450)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:376)
	at org.apache.spark.scheduler.DAGScheduler.handleJobGroupCancelled(DAGScheduler.scala:1331)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.doOnReceive(DAGScheduler.scala:3751)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onReceive(DAGScheduler.scala:3724)
	at org.apache.spark.scheduler.DAGSc

In [41]:

# Read the facility dimension
df_facility = spark.table("dbo.dimfacility")

# Standardize Facility_ID
df_facility_clean = df_facility.withColumn(
    "Facility_ID",
    F.upper(F.trim(F.col("Facility_ID")))
)

# Check for duplicate keys after standardization
print(
    "Facility rows:",
    df_facility_clean.count()
)

print(
    "Unique standardized Facility IDs:",
    df_facility_clean.select("Facility_ID")
    .distinct()
    .count()
)


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 43, Finished, Available, Finished, False)

Facility rows: 25
Unique standardized Facility IDs: 25


In [42]:

df_facility.printSchema()
display(df_facility.limit(5))


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 44, Finished, Available, Finished, False)

root
 |-- Facility_ID: string (nullable = true)
 |-- Facility_Name: string (nullable = true)
 |-- Facility_Type: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP_Code: integer (nullable = true)
 |-- County: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Licensed_Beds: string (nullable = true)
 |-- CMS_Style_Rating: string (nullable = true)
 |-- Open_Date: string (nullable = true)



SynapseWidget(Synapse.DataFrame, e36145fe-3345-45ec-9ebe-db40882597da)

In [44]:

# Inspect DimFacility columns
df_facility = spark.table("dbo.dimfacility")

df_facility.printSchema()

display(df_facility.limit(5))


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 46, Finished, Available, Finished, False)

root
 |-- Facility_ID: string (nullable = true)
 |-- Facility_Name: string (nullable = true)
 |-- Facility_Type: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP_Code: integer (nullable = true)
 |-- County: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Licensed_Beds: string (nullable = true)
 |-- CMS_Style_Rating: string (nullable = true)
 |-- Open_Date: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 3fdf5266-6d54-4e6b-b90f-5332026d13fc)

In [49]:

df_facility = spark.table("dbo.dimfacility")

df_facility.printSchema()

display(df_facility.limit(5))


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 51, Finished, Cancelled, Cancelled, False)

root
 |-- Facility_ID: string (nullable = true)
 |-- Facility_Name: string (nullable = true)
 |-- Facility_Type: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP_Code: integer (nullable = true)
 |-- County: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Licensed_Beds: string (nullable = true)
 |-- CMS_Style_Rating: string (nullable = true)
 |-- Open_Date: string (nullable = true)



Py4JJavaError: An error occurred while calling z:com.microsoft.spark.notebook.visualization.display.getDisplayResultForIPython.
: org.apache.spark.SparkException: [SPARK_JOB_CANCELLED] Job 327 cancelled part of cancelled job group 51 SQLSTATE: XXKDA
	at org.apache.spark.errors.SparkCoreErrors$.sparkJobCancelled(SparkCoreErrors.scala:222)
	at org.apache.spark.scheduler.DAGScheduler.handleJobCancellation(DAGScheduler.scala:3428)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleJobGroupCancelled$7(DAGScheduler.scala:1331)
	at scala.runtime.java8.JFunction1$mcVI$sp.apply(JFunction1$mcVI$sp.scala:18)
	at scala.collection.mutable.HashSet$Node.foreach(HashSet.scala:450)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:376)
	at org.apache.spark.scheduler.DAGScheduler.handleJobGroupCancelled(DAGScheduler.scala:1331)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.doOnReceive(DAGScheduler.scala:3751)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onRe

In [50]:

from pyspark.sql import functions as F

# Read existing tables
df_fact = spark.table("dbo.factappointments")
df_facility = spark.table("dbo.dimfacility")

# Standardize Facility_ID in both tables
df_fact_clean = df_fact.withColumn(
    "Facility_ID",
    F.upper(F.trim(F.col("Facility_ID")))
)

df_facility_clean = df_facility.withColumn(
    "Facility_ID",
    F.upper(F.trim(F.col("Facility_ID")))
)

# Validate counts
print("Appointments:", df_fact_clean.count())
print("Facilities:", df_facility_clean.count())
print(
    "Unique Facility IDs:",
    df_facility_clean.select("Facility_ID").distinct().count()
)


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 52, Finished, Available, Finished, False)

Appointments: 10000
Facilities: 25
Unique Facility IDs: 25


In [52]:

from pyspark.sql import functions as F

# Read the existing facility dimension
df_facility = spark.table("dbo.dimfacility")

# Standardize Facility IDs
df_facility_clean = df_facility.withColumn(
    "Facility_ID",
    F.upper(F.trim(F.col("Facility_ID")))
)

# Create an Unknown Facility record
# using the existing 12-column schema
unknown_facility = spark.createDataFrame(
    [(
        "FAC-999",
        "Unknown Facility",
        "Unknown",
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        None
    )],
    schema=df_facility_clean.schema
)

# Add the placeholder only if it is absent
if df_facility_clean.filter(
    F.col("Facility_ID") == "FAC-999"
).count() == 0:
    df_facility_final = df_facility_clean.unionByName(
        unknown_facility
    )
else:
    df_facility_final = df_facility_clean

# Validate before saving
assert df_facility_final.count() == 26
assert (
    df_facility_final.select("Facility_ID")
    .distinct().count() == 26
)

# Save as a NEW table
df_facility_final.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable("dbo.DimFacility_Clean")

print("DimFacility_Clean created successfully!")
print("Total facilities:", df_facility_final.count())


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 54, Finished, Available, Finished, False)

AnalysisException: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `chimcobldhq2ai35c5m78q33c5p6anqedt9mgrrnbt0msobcf5q6iorj4l46aobcehk66obiclfkoobbclk6utbjckim8ojf`.`DimFacility_Clean` because it already exists.
Choose a different name, drop or replace the existing object, or add the IF NOT EXISTS clause to tolerate pre-existing objects. SQLSTATE: 42P07

In [53]:

# Read existing appointments
df_fact = spark.table("dbo.factappointments")

# Standardize Facility_ID
df_fact_clean = df_fact.withColumn(
    "Facility_ID",
    F.upper(F.trim(F.col("Facility_ID")))
)

# Verify row count
assert df_fact_clean.count() == 10000

# Verify unique appointment IDs
assert (
    df_fact_clean.select("Encounter_ID")
    .distinct().count() == 10000
)

# Save as a NEW table
df_fact_clean.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable("dbo.FactAppointments_Clean")

print("FactAppointments_Clean created successfully!")
print("Total appointments:", df_fact_clean.count())


StatementMeta(, 5f2c1005-344a-40c4-8251-00b2357d62d4, 55, Finished, Available, Finished, False)

FactAppointments_Clean created successfully!
Total appointments: 10000


In [1]:

# Read the original diagnosis table
df_diagnosis = spark.table("dbo.CodeDiagnosis")

# Show the structure
df_diagnosis.printSchema()

# Preview diagnosis records
display(df_diagnosis.limit(10))

# Count all rows
print("Total diagnosis rows:", df_diagnosis.count())


StatementMeta(, f526329d-eaac-4639-9a5d-dfdf0a7a3d97, 3, Finished, Available, Finished, False)

root
 |-- Diagnosis_Code: string (nullable = true)
 |-- Plain_English_Description: string (nullable = true)



SynapseWidget(Synapse.DataFrame, f88825ee-42ff-4791-89fa-d84f595ecb1c)

Total diagnosis rows: 12


In [2]:

from pyspark.sql import functions as F

# Check missing or blank values
missing_values = df_diagnosis.select([
    F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_diagnosis.columns
])

display(missing_values)


StatementMeta(, f526329d-eaac-4639-9a5d-dfdf0a7a3d97, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5c450cca-36f6-4f53-adcd-8c04c21343ef)

In [3]:

# Count original records
total_rows = df_diagnosis.count()

# Count distinct complete records
distinct_rows = df_diagnosis.distinct().count()

# Count unique diagnosis codes
unique_codes = (
    df_diagnosis
    .select("Diagnosis_Code")
    .distinct()
    .count()
)

print("Total diagnosis rows:", total_rows)
print("Distinct complete rows:", distinct_rows)
print("Unique diagnosis codes:", unique_codes)
print("Exact duplicate rows:", total_rows - distinct_rows)


StatementMeta(, f526329d-eaac-4639-9a5d-dfdf0a7a3d97, 5, Finished, Available, Finished, False)

Total diagnosis rows: 12
Distinct complete rows: 12
Unique diagnosis codes: 12
Exact duplicate rows: 0


In [4]:

# Remove exact duplicates in memory
df_diagnosis_distinct = df_diagnosis.dropDuplicates()

# Standardize diagnosis codes
df_diagnosis_clean = (
    df_diagnosis_distinct
    .withColumn(
        "Diagnosis_Code",
        F.upper(F.trim(F.col("Diagnosis_Code")))
    )
    .withColumn(
        "Plain_English_Description",
        F.trim(F.col("Plain_English_Description"))
    )
)

# Check for repeated codes after standardization
conflicts = (
    df_diagnosis_clean
    .groupBy("Diagnosis_Code")
    .count()
    .filter(F.col("count") > 1)
)

print("Conflicting diagnosis codes:", conflicts.count())

display(conflicts)


StatementMeta(, f526329d-eaac-4639-9a5d-dfdf0a7a3d97, 6, Finished, Available, Finished, False)

Conflicting diagnosis codes: 0


SynapseWidget(Synapse.DataFrame, 2702a9a1-541a-42c9-a862-1c43ee86c08f)

In [5]:

# Check missing or blank diagnosis codes
missing_codes = df_diagnosis_clean.filter(
    F.col("Diagnosis_Code").isNull() |
    (F.col("Diagnosis_Code") == "")
).count()

# Check final record counts
clean_rows = df_diagnosis_clean.count()

unique_clean_codes = (
    df_diagnosis_clean
    .select("Diagnosis_Code")
    .distinct()
    .count()
)

print("Clean diagnosis rows:", clean_rows)
print("Unique diagnosis codes:", unique_clean_codes)
print("Missing diagnosis codes:", missing_codes)
print("Conflicting codes:", conflicts.count())


StatementMeta(, f526329d-eaac-4639-9a5d-dfdf0a7a3d97, 7, Finished, Available, Finished, False)

Clean diagnosis rows: 12
Unique diagnosis codes: 12
Missing diagnosis codes: 0
Conflicting codes: 0


In [6]:

# Safety checks
assert missing_codes == 0, "Missing diagnosis codes found"
assert conflicts.count() == 0, "Conflicting diagnosis codes found"
assert clean_rows == unique_clean_codes, "Duplicate codes found"

# Create a new Delta table
df_diagnosis_clean.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable("dbo.DimDiagnosis")

print("DimDiagnosis created successfully!")
print("Total diagnosis codes:", clean_rows)


StatementMeta(, f526329d-eaac-4639-9a5d-dfdf0a7a3d97, 8, Finished, Available, Finished, False)

DimDiagnosis created successfully!
Total diagnosis codes: 12


In [1]:

from pyspark.sql import functions as F

# Read cleaned appointments
df_fact = spark.table("dbo.factappointments_clean")

# Inspect Encounter_Date
df_fact.select("Encounter_Date").printSchema()

# Preview appointment dates
display(
    df_fact.select("Encounter_Date").limit(10)
)

# Find earliest and latest appointment dates
df_fact.select(
    F.min(F.to_date("Encounter_Date")).alias("Earliest_Date"),
    F.max(F.to_date("Encounter_Date")).alias("Latest_Date")
).show()


StatementMeta(, 9ffe7cba-3114-4b73-87b0-cce28154e391, 3, Finished, Available, Finished, False)

root
 |-- Encounter_Date: string (nullable = true)



SynapseWidget(Synapse.DataFrame, a1cbe569-1eaa-4253-b982-ed2453cc66f4)

+-------------+-----------+
|Earliest_Date|Latest_Date|
+-------------+-----------+
|   2022-12-31| 2025-12-27|
+-------------+-----------+



In [2]:

from pyspark.sql import functions as F

# Define calendar boundaries
start_date = "2023-01-01"
end_date = "2025-12-31"

# Generate one row per calendar day
df_dates = spark.sql(f"""
    SELECT explode(
        sequence(
            to_date('{start_date}'),
            to_date('{end_date}'),
            interval 1 day
        )
    ) AS Date
""")

print("Total calendar days:", df_dates.count())

display(df_dates.limit(10))


StatementMeta(, 9ffe7cba-3114-4b73-87b0-cce28154e391, 4, Finished, Available, Finished, False)

Total calendar days: 1096


SynapseWidget(Synapse.DataFrame, 9b2bd8c4-a70b-4a92-9ee4-fd09ce87e473)

In [3]:

from pyspark.sql import functions as F

df_dimdate = (
    df_dates
    .withColumn("Year", F.year("Date"))
    .withColumn("Quarter", F.quarter("Date"))
    .withColumn("Month_Number", F.month("Date"))
    .withColumn(
        "Month_Name",
        F.date_format("Date", "MMMM")
    )
    .withColumn(
        "Month_Year",
        F.date_format("Date", "MMM yyyy")
    )
    .withColumn(
        "Quarter_Year",
        F.concat(
            F.lit("Q"),
            F.quarter("Date"),
            F.lit(" "),
            F.year("Date")
        )
    )
    .withColumn(
        "Quarter_Year_Sort",
        F.year("Date") * 10 + F.quarter("Date")
    )
    .withColumn(
        "Month_Year_Sort",
        F.year("Date") * 100 + F.month("Date")
    )
    .withColumn(
        "Quarter_Start",
        F.expr("date_trunc('quarter', Date)").cast("date")
    )
    .withColumn(
        "Quarter_End",
        F.last_day(
            F.add_months(
                F.expr("date_trunc('quarter', Date)"),
                2
            )
        )
    )
)

display(df_dimdate.limit(10))


StatementMeta(, 9ffe7cba-3114-4b73-87b0-cce28154e391, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 88c3558f-4047-418e-a48c-ee34ecd7b40f)

In [4]:

# Count calendar rows
total_dates = df_dimdate.count()

# Count unique dates
unique_dates = (
    df_dimdate.select("Date").distinct().count()
)

# Check missing dates
missing_dates = df_dimdate.filter(
    F.col("Date").isNull()
).count()

# Check number of quarters
total_quarters = (
    df_dimdate.select("Quarter_Year")
    .distinct().count()
)

print("Total dates:", total_dates)
print("Unique dates:", unique_dates)
print("Missing dates:", missing_dates)
print("Total quarters:", total_quarters)

assert total_dates == 1096
assert unique_dates == 1096
assert missing_dates == 0
assert total_quarters == 12

print("DimDate validation passed!")


StatementMeta(, 9ffe7cba-3114-4b73-87b0-cce28154e391, 6, Finished, Available, Finished, False)

Total dates: 1096
Unique dates: 1096
Missing dates: 0
Total quarters: 12
DimDate validation passed!


In [5]:

# Save DimDate as a new Delta table
df_dimdate.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable("dbo.DimDate")

print("DimDate created successfully!")
print("Total calendar records:", df_dimdate.count())


StatementMeta(, 9ffe7cba-3114-4b73-87b0-cce28154e391, 7, Finished, Available, Finished, False)

DimDate created successfully!
Total calendar records: 1096


In [6]:

from pyspark.sql import functions as F

# Read the existing cleaned fact table
df_fact = spark.table("dbo.factappointments_clean")

# Convert both date formats
df_dates_fixed = df_fact.withColumn(
    "Encounter_Date_Clean",
    F.coalesce(
        F.to_date("Encounter_Date", "yyyy-MM-dd"),
        F.to_date("Encounter_Date", "dd-MM-yyyy")
    )
)

# Check results
print("Total appointments:", df_dates_fixed.count())

print(
    "Dates successfully converted:",
    df_dates_fixed.filter(
        F.col("Encounter_Date_Clean").isNotNull()
    ).count()
)

print(
    "Dates still invalid:",
    df_dates_fixed.filter(
        F.col("Encounter_Date_Clean").isNull()
    ).count()
)

display(
    df_dates_fixed.select(
        "Encounter_ID",
        "Encounter_Date",
        "Encounter_Date_Clean"
    ).limit(20)
)


StatementMeta(, 9ffe7cba-3114-4b73-87b0-cce28154e391, 8, Finished, Available, Finished, False)

Total appointments: 10000
Dates successfully converted: 9659
Dates still invalid: 341


SynapseWidget(Synapse.DataFrame, e24e7869-47b7-413f-8d09-f9762a5159a1)

In [7]:

from pyspark.sql import functions as F

# Display dates that failed conversion
df_failed_dates = df_dates_fixed.filter(
    F.col("Encounter_Date_Clean").isNull()
)

print("Unconverted dates:", df_failed_dates.count())

display(
    df_failed_dates.select(
        "Encounter_ID",
        "Encounter_Date"
    ).orderBy("Encounter_Date").limit(50)
)


StatementMeta(, 9ffe7cba-3114-4b73-87b0-cce28154e391, 9, Finished, Available, Finished, False)

Unconverted dates: 341


SynapseWidget(Synapse.DataFrame, 5d4fdb4e-0a5d-4e73-9222-23cf450b03a8)

In [8]:

# Group original dates by their visible format
df_fact.select(
    F.when(
        F.col("Encounter_Date").rlike(
            r"^\d{4}-\d{2}-\d{2}$"
        ),
        "yyyy-MM-dd"
    )
    .when(
        F.col("Encounter_Date").rlike(
            r"^\d{2}-\d{2}-\d{4}$"
        ),
        "dd-MM-yyyy or MM-dd-yyyy"
    )
    .when(
        F.col("Encounter_Date").isNull(),
        "NULL"
    )
    .otherwise("Other format")
    .alias("Date_Format")
).groupBy("Date_Format").count().show()


StatementMeta(, 9ffe7cba-3114-4b73-87b0-cce28154e391, 10, Finished, Available, Finished, False)

+--------------------+-----+
|         Date_Format|count|
+--------------------+-----+
|          yyyy-MM-dd| 9499|
|        Other format|  341|
|dd-MM-yyyy or MM-...|  160|
+--------------------+-----+



In [9]:

display(
    df_failed_dates.select(
        "Encounter_ID",
        "Encounter_Date",
        F.length("Encounter_Date").alias("Length"),
        F.length(
            F.trim("Encounter_Date")
        ).alias("Trimmed_Length")
    ).limit(30)
)


StatementMeta(, 9ffe7cba-3114-4b73-87b0-cce28154e391, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a341eff2-1b30-41c6-933d-23892091358a)

In [10]:

from pyspark.sql import functions as F

df_fact = spark.table("dbo.factappointments_clean")

# Remove leading and trailing spaces
raw_date = F.trim(F.col("Encounter_Date"))

# Convert different date formats
df_dates_fixed = df_fact.withColumn(
    "Encounter_Date_Clean",
    F.coalesce(
        F.try_to_timestamp(
            raw_date,
            F.lit("yyyy-MM-dd HH:mm:ss")
        ).cast("date"),

        F.try_to_timestamp(
            raw_date,
            F.lit("yyyy-MM-dd")
        ).cast("date"),

        F.try_to_timestamp(
            raw_date,
            F.lit("dd-MM-yyyy")
        ).cast("date"),

        F.try_to_timestamp(
            raw_date,
            F.lit("M/d/yy")
        ).cast("date"),

        F.try_to_timestamp(
            raw_date,
            F.lit("M/d/yyyy")
        ).cast("date")
    )
)

# Validate conversion
total = df_dates_fixed.count()

converted = df_dates_fixed.filter(
    F.col("Encounter_Date_Clean").isNotNull()
).count()

invalid = total - converted

print("Total appointments:", total)
print("Successfully converted:", converted)
print("Still invalid:", invalid)


StatementMeta(, 9ffe7cba-3114-4b73-87b0-cce28154e391, 12, Finished, Available, Finished, False)

Total appointments: 10000
Successfully converted: 10000
Still invalid: 0


In [11]:

df_dates_fixed.groupBy(
    F.when(
        F.col("Encounter_Date_Clean").isNull(),
        "Invalid"
    )
    .when(
        F.col("Encounter_Date_Clean") <
        F.lit("2023-01-01").cast("date"),
        "Before 2023"
    )
    .when(
        F.col("Encounter_Date_Clean") >
        F.lit("2025-12-31").cast("date"),
        "After 2025"
    )
    .otherwise("Within 2023-2025")
    .alias("Date_Status")
).count().show()


StatementMeta(, 9ffe7cba-3114-4b73-87b0-cce28154e391, 13, Finished, Available, Finished, False)

+----------------+-----+
|     Date_Status|count|
+----------------+-----+
|     Before 2023|   11|
|Within 2023-2025| 9989|
+----------------+-----+



In [12]:

from pyspark.sql import functions as F

# Keep the original date text for auditing
df_fact_final = (
    df_dates_fixed
    .withColumnRenamed(
        "Encounter_Date",
        "Encounter_Date_Original"
    )
    .withColumnRenamed(
        "Encounter_Date_Clean",
        "Encounter_Date"
    )
)

# Verify all appointments and dates
assert df_fact_final.count() == 10000

assert (
    df_fact_final
    .filter(F.col("Encounter_Date").isNull())
    .count() == 0
)

assert (
    df_fact_final
    .select("Encounter_ID")
    .distinct()
    .count() == 10000
)

# Save without overwriting existing tables
df_fact_final.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable("dbo.FactAppointments_Final")

print("FactAppointments_Final created successfully!")
print("Total appointments:", df_fact_final.count())

df_fact_final.select(
    "Encounter_ID",
    "Encounter_Date_Original",
    "Encounter_Date"
).show(10, truncate=False)


StatementMeta(, 9ffe7cba-3114-4b73-87b0-cce28154e391, 14, Finished, Available, Finished, False)

FactAppointments_Final created successfully!
Total appointments: 10000
+------------+-----------------------+--------------+
|Encounter_ID|Encounter_Date_Original|Encounter_Date|
+------------+-----------------------+--------------+
|ENC-000680  |2023-07-03             |2023-07-03    |
|ENC-000737  |2024-06-05             |2024-06-05    |
|ENC-000911  |2025-04-09             |2025-04-09    |
|ENC-001012  |2025-12-06             |2025-12-06    |
|ENC-001357  |2023-10-10             |2023-10-10    |
|ENC-003342  |2025-07-31             |2025-07-31    |
|ENC-004034  |2023-11-11             |2023-11-11    |
|ENC-004109  |2023-11-18             |2023-11-18    |
|ENC-004187  |2024-12-20             |2024-12-20    |
|ENC-004470  |2024-02-16             |2024-02-16    |
+------------+-----------------------+--------------+
only showing top 10 rows


In [1]:

from pyspark.sql import functions as F

df = df.withColumn(
    "Visit_Type",
    F.when(
        F.lower(
            F.regexp_replace(
                F.trim(F.col("Visit_Type")),
                r"[\s-]+",
                ""
            )
        ) == "followup",
        F.lit("Follow-Up")
    ).otherwise(F.col("Visit_Type"))
)


StatementMeta(, 74438e3e-6526-4f57-8868-2ce034ba2ab9, 3, Finished, Available, Finished, False)

NameError: name 'df' is not defined

In [2]:

df = spark.table("dbo.FactAppointments_Final")

print("Total appointments:", df.count())
df.select("Visit_Type").distinct().show(truncate=False)


StatementMeta(, 74438e3e-6526-4f57-8868-2ce034ba2ab9, 4, Finished, Available, Finished, False)

Total appointments: 10000
+------------------+
|Visit_Type        |
+------------------+
|Follow-Up         |
|Specialist Consult|
|Behavioral Health |
|Primary Care      |
|Urgent Care       |
|Care Coordination |
|follow up         |
|Preventive Visit  |
|Telehealth        |
+------------------+



In [3]:

from pyspark.sql import functions as F

df_clean = df.withColumn(
    "Visit_Type",
    F.when(
        F.lower(
            F.regexp_replace(
                F.trim(F.col("Visit_Type")),
                r"[\s-]+",
                ""
            )
        ) == "followup",
        F.lit("Follow-Up")
    ).otherwise(F.col("Visit_Type"))
)

df_clean.groupBy("Visit_Type").count().orderBy("Visit_Type").show(
    truncate=False
)


StatementMeta(, 74438e3e-6526-4f57-8868-2ce034ba2ab9, 5, Finished, Available, Finished, False)

+------------------+-----+
|Visit_Type        |count|
+------------------+-----+
|Behavioral Health |1230 |
|Care Coordination |1233 |
|Follow-Up         |1304 |
|Preventive Visit  |1224 |
|Primary Care      |1268 |
|Specialist Consult|1260 |
|Telehealth        |1219 |
|Urgent Care       |1262 |
+------------------+-----+



In [4]:

from pyspark.sql import functions as F

df_clean = df.withColumn(
    "Visit_Type",
    F.when(
        F.lower(
            F.regexp_replace(
                F.trim(F.col("Visit_Type")),
                r"[\s-]+",
                ""
            )
        ) == "followup",
        F.lit("Follow-Up")
    ).otherwise(F.col("Visit_Type"))
)

df_clean.groupBy("Visit_Type").count().orderBy("Visit_Type").show(
    truncate=False
)


StatementMeta(, 74438e3e-6526-4f57-8868-2ce034ba2ab9, 6, Finished, Available, Finished, False)

+------------------+-----+
|Visit_Type        |count|
+------------------+-----+
|Behavioral Health |1230 |
|Care Coordination |1233 |
|Follow-Up         |1304 |
|Preventive Visit  |1224 |
|Primary Care      |1268 |
|Specialist Consult|1260 |
|Telehealth        |1219 |
|Urgent Care       |1262 |
+------------------+-----+



In [5]:

print("Original rows:", df.count())
print("Cleaned rows:", df_clean.count())

df_clean.filter(
    F.col("Visit_Type") == "Follow-Up"
).count()


StatementMeta(, 74438e3e-6526-4f57-8868-2ce034ba2ab9, 7, Finished, Available, Finished, False)

Original rows: 10000
Cleaned rows: 10000


1304

In [6]:

# Back up the original appointment table

df.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable("dbo.FactAppointments_Backup")


StatementMeta(, 74438e3e-6526-4f57-8868-2ce034ba2ab9, 8, Finished, Available, Finished, False)

In [7]:

backup_count = spark.table(
    "dbo.FactAppointments_Backup"
).count()

print("Backup appointments:", backup_count)


StatementMeta(, 74438e3e-6526-4f57-8868-2ce034ba2ab9, 9, Finished, Available, Finished, False)

Backup appointments: 10000


In [8]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Access the existing Delta table
appointments = DeltaTable.forName(
    spark,
    "dbo.FactAppointments_Final"
)

# Standardize the Follow-Up category in place
appointments.update(
    condition="""
        lower(
            regexp_replace(
                trim(Visit_Type),
                '[\\\\s-]+',
                ''
            )
        ) = 'followup'
    """,
    set={
        "Visit_Type": "'Follow-Up'"
    }
)

print("Visit Type update completed.")


StatementMeta(, 74438e3e-6526-4f57-8868-2ce034ba2ab9, 10, Finished, Available, Finished, False)

Visit Type update completed.


In [9]:

from pyspark.sql import functions as F

updated_df = spark.table("dbo.FactAppointments_Final")

print("Total appointments:", updated_df.count())
print("Distinct visit types:",
      updated_df.select("Visit_Type").distinct().count())

updated_df.groupBy("Visit_Type") \
    .count() \
    .orderBy("Visit_Type") \
    .show(truncate=False)


StatementMeta(, 74438e3e-6526-4f57-8868-2ce034ba2ab9, 11, Finished, Available, Finished, False)

Total appointments: 10000
Distinct visit types: 8
+------------------+-----+
|Visit_Type        |count|
+------------------+-----+
|Behavioral Health |1230 |
|Care Coordination |1233 |
|Follow-Up         |1304 |
|Preventive Visit  |1224 |
|Primary Care      |1268 |
|Specialist Consult|1260 |
|Telehealth        |1219 |
|Urgent Care       |1262 |
+------------------+-----+



In [10]:

df = spark.table("dbo.FactAppointments_Final")

df.groupBy("Visit_Mode") \
    .count() \
    .orderBy("Visit_Mode") \
    .show(truncate=False)


StatementMeta(, 74438e3e-6526-4f57-8868-2ce034ba2ab9, 12, Finished, Available, Finished, False)

+-----------+-----+
|Visit_Mode |count|
+-----------+-----+
|Home Visit |3289 |
|In Person  |3400 |
|Telehealth |3256 |
|tele-health|55   |
+-----------+-----+



In [11]:

from delta.tables import DeltaTable

appointments = DeltaTable.forName(
    spark,
    "dbo.FactAppointments_Final"
)

appointments.update(
    condition="""
        lower(
            regexp_replace(
                trim(Visit_Mode),
                '[^a-zA-Z]',
                ''
            )
        ) = 'telehealth'
    """,
    set={
        "Visit_Mode": "'Telehealth'"
    }
)

print("Visit Mode update completed.")


StatementMeta(, 74438e3e-6526-4f57-8868-2ce034ba2ab9, 13, Finished, Available, Finished, False)

Visit Mode update completed.


In [12]:

updated_df = spark.table("dbo.FactAppointments_Final")

print("Total appointments:", updated_df.count())

updated_df.groupBy("Visit_Mode") \
    .count() \
    .orderBy("Visit_Mode") \
    .show(truncate=False)


StatementMeta(, 74438e3e-6526-4f57-8868-2ce034ba2ab9, 14, Finished, Available, Finished, False)

Total appointments: 10000
+----------+-----+
|Visit_Mode|count|
+----------+-----+
|Home Visit|3289 |
|In Person |3400 |
|Telehealth|3311 |
+----------+-----+

